# FlavourBench evidence and publication-readiness audit

**Purpose.** Reproduce the denominators behind the retrospective Season 0 paper, inspect endpoint uncertainty and judge survival, and keep the 2026-07-28 frontier refresh separate from the frozen pilot.

**Decision rule.** A technically complete provider run is not sufficient for a culinary leaderboard. Endpoint ranking additionally requires the declared comparison minimum, a usable comparison graph, qualified human criterion evidence, reviewed items, and a reproducible Epicure release.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ANALYSIS = ROOT / (
    "artifacts/season0/analysis-v6/"
    "season0-automated-analysis-ab45eff77098a97fc05ef7ee5ca689b00724381e4bd8c6f7e4dd60c86fb61d97.json"
)
MANIFEST = ROOT / (
    "artifacts/season0/manifests/"
    "season0-model-manifest-3919def66686b4bd939c94cdd89659f63ae2afbbf03288413129e2ea8d6b83d2.json"
)
PAPER_DATA = ROOT.parent / "paper/flavourbench/generated/pilot"

analysis = json.loads(ANALYSIS.read_text())
manifest = json.loads(MANIFEST.read_text())
print({"analysis": ANALYSIS.name, "manifest": MANIFEST.name})

{'analysis': 'season0-automated-analysis-ab45eff77098a97fc05ef7ee5ca689b00724381e4bd8c6f7e4dd60c86fb61d97.json', 'manifest': 'season0-model-manifest-3919def66686b4bd939c94cdd89659f63ae2afbbf03288413129e2ea8d6b83d2.json'}


## 1. Integrity and fixed denominators

The analysis artifact is content addressed. The check below removes the self-referential digest field and reproduces its canonical SHA-256.

In [2]:
def canonical_sha256(document):
    payload = dict(document)
    expected = payload.pop("artifact_sha256")
    rendered = json.dumps(
        payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    ).encode()
    return expected, hashlib.sha256(rendered).hexdigest()


expected, observed = canonical_sha256(analysis)
assert observed == expected == "ab45eff77098a97fc05ef7ee5ca689b00724381e4bd8c6f7e4dd60c86fb61d97"
counts = analysis["counts"]
integrity = pd.Series(
    {
        "attempted target arms": counts["scored_arms"],
        "recorded Epicure calls": counts["recorded_epicure_calls_including_partial"],
        "planned comparisons": counts["comparison_manifest_rows"],
        "judgment records": counts["judgment_records"],
        "primary consensuses": counts["consensus_available"],
    }
)
integrity

attempted target arms      2880
recorded Epicure calls     1387
planned comparisons        2160
judgment records          14184
primary consensuses         393
dtype: int64

## 2. Endpoint comparison uncertainty

Every endpoint falls below the manuscript's 100-comparison reporting threshold. The table shows the task-bootstrap ridge diagnostic, not a validated culinary-quality ranking.

In [3]:
model = pd.read_csv(PAPER_DATA / "pilot-model-uncertainty.csv")
model["interval_width"] = model["bootstrap_upper"] - model["bootstrap_lower"]
model["below_threshold"] = model["n"] < 100
assert model["below_threshold"].all()
assert model["complete_separation"].sum() == 1
model[
    [
        "model",
        "n",
        "wins",
        "ties",
        "losses",
        "bootstrap_median",
        "bootstrap_lower",
        "bootstrap_upper",
        "highest_resample_fraction",
        "complete_separation",
    ]
].sort_values("bootstrap_median", ascending=False)

,model,n,wins,ties,losses,bootstrap_median,bootstrap_lower,bootstrap_upper,highest_resample_fraction,complete_separation
7,OpenAI gpt-oss-120b,14,13,1,0,1911.022350,1446.193613,3802.466212,0.493401,False
0,Anthropic Claude Sonnet 4.6,4,3,0,1,1714.271497,890.717810,3561.537776,0.241624,False
2,Anthropic Claude Fable 5,18,16,2,0,1650.787767,1087.456403,3826.824981,0.228426,False
10,OpenAI GPT-5.6 Terra,27,23,3,1,1614.207422,1179.996029,3940.925512,0.034518,False
3,OpenAI GPT-5.6 Sol,29,22,2,5,1322.172383,806.779917,1785.739611,0.002030,False
1,Anthropic Claude Opus 4.8,38,23,5,10,1170.813966,590.536237,1707.792966,0.000000,False
11,Google Gemini 3.1 Pro Preview,13,8,1,4,1122.592938,539.567602,1610.068405,0.000000,False
4,Qwen3 Next 80B A3B,29,12,2,15,909.830441,224.260050,1297.032914,0.000000,False
5,Qwen3 VL 235B A22B,27,10,2,15,835.841664,119.946528,1250.158697,0.000000,False
9,MiniMax M2.5,33,10,2,21,830.640103,211.085817,1206.709131,0.000000,False


## 3. Judge survival and paired estimand sensitivity

Orientation replication is a real quality control, but here it creates strong, model-dependent missingness. The paired result also changes with weighting and admission.

In [4]:
judge = pd.read_csv(PAPER_DATA / "pilot-measurement-integrity.csv")
judge = judge[judge["panel"] == "judge"].pivot(index="label", columns="metric", values="rate")
uplift = pd.read_csv(PAPER_DATA / "pilot-epicure-robustness.csv")
reliability = pd.read_csv(PAPER_DATA / "pilot-condition-reliability.csv").iloc[0]
display(judge.sort_values("eligible_vote_yield", ascending=False))
display(uplift[["label", "estimate", "lower", "upper", "wins", "ties", "losses", "n"]])
print(
    {
        "on_minus_off_success_risk": reliability["realized_success_proportion_difference"],
        "interval": [
            reliability["lower_95_task_bootstrap"],
            reliability["upper_95_task_bootstrap"],
        ],
    }
)

metric,agreement_given_completion,both_orientations_complete,eligible_vote_yield
label,,,
Claude Haiku 4.5,0.609909,0.964334,0.588156
Claude Sonnet 4.6,0.731680,0.596904,0.406460
Devstral 2 123B,0.816754,0.385599,0.278600
Qwen3 Next 80B,1.000000,0.004711,0.004038


,label,estimate,lower,upper,wins,ties,losses,n
0,Cell weighted (primary),0.427386,0.388430,0.465910,31,144,66,241
1,Equal task means,0.418336,0.369606,0.465804,31,144,66,241
2,Equal endpoint means,0.487282,0.450157,0.521184,31,144,66,241
3,"Primary, item removed",0.425847,0.385919,0.463987,31,139,66,236
4,Cross-family-admitted subset,0.503759,0.465035,0.544777,16,102,15,133


{'on_minus_off_success_risk': np.float64(-0.1291666666666666), 'interval': [np.float64(-0.1604166666666666), np.float64(-0.1)]}


## 4. Frontier-refresh availability evidence

The refresh is append-only and unranked. Failed infrastructure or entitlement checks are not converted into model-quality losses.

In [5]:
summaries = sorted(
    (ROOT / "artifacts/frontier-refresh/2026-07-28").glob(
        "compatibility*/frontier-refresh-contract-summary-*.json"
    )
)
refresh_rows = []
for path in summaries:
    document = json.loads(path.read_text())
    for arm in document["artifacts"]:
        refresh_rows.append(
            {
                "batch": path.parent.name,
                "provider": arm["provider"],
                "model": arm["display_name"],
                "status": arm["status"],
                "provider_calls": arm["provider_calls"],
                "epicure_calls": arm["epicure_calls"],
                "error_type": arm.get("error_type"),
            }
        )
refresh = pd.DataFrame(refresh_rows)
refresh

,batch,provider,model,status,provider_calls,epicure_calls,error_type
0,compatibility,bedrock,Anthropic Claude Opus 5,failed,0,0,ConnectError
1,compatibility,bedrock,Anthropic Claude Sonnet 5,failed,0,0,ConnectError
2,compatibility,openrouter,MoonshotAI Kimi K3,failed,0,0,HTTPStatusError
3,compatibility,openrouter,Z.ai GLM 5.2,failed,0,0,HTTPStatusError
4,compatibility-us-west-2-v1,bedrock,Anthropic Claude Opus 5,failed,0,0,AccessDeniedException
5,compatibility-us-west-2-v1,bedrock,Anthropic Claude Sonnet 5,failed,0,0,AccessDeniedException
6,compatibility-us-west-2-v2,bedrock,Anthropic Claude Opus 5,failed,0,0,AccessDeniedException
7,compatibility-us-west-2-v2,bedrock,Anthropic Claude Sonnet 5,failed,0,0,AccessDeniedException
8,compatibility-v2,bedrock,Anthropic Claude Opus 5,failed,0,0,HTTPStatusError
9,compatibility-v2,bedrock,Anthropic Claude Sonnet 5,failed,0,0,HTTPStatusError


## 5. Publication decision

The retrospective evidence is suitable for a measurement-audit paper, not a quality leaderboard. Four blockers are decisive:

1. no endpoint reaches 100 valid automated-consensus comparisons;
2. one endpoint is completely separated and the ordering is unstable under task resampling;
3. no qualified independent human criterion cohort or completed item review exists; and
4. the executed Epicure bundle is content addressed but lacks a redistributable, independently reproducible lineage.

The newest-model refresh cannot repair those issues retroactively. Opus 5 and Sonnet 5 were discovered as exact Bedrock global profiles, but the account denied inference entitlement. Kimi K3 and GLM 5.2 were frozen to exact OpenRouter endpoints, but the provider account did not admit the contract-smoke batch. Those are route-availability observations only.